# 跨文档智能审计与对比系统

本项目是一款基于 Ollama (Qwen3-Coder:30B) 开发的高级文档处理工具，专门用于解决超长文档的深度分析及多文档间的横向对比难题。系统集成了滑动窗口切片、多级级联摘要和跨文档立场审计功能，能够从海量非结构化文本中自动提取核心要点、情感倾向及矛盾差异。

In [ ]:
# 检查 AMD GPU 是否可用
import torch
print(f'ROCm 可用: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "未检测到"}')

In [ ]:
# author:GuanHua Yu
import httpx
import json
import asyncio
import ipywidgets as widgets

OLLAMA_URL = "http://open-webui-ollama.open-webui:11434/api/generate"
MODEL_NAME = "qwen3-coder:30b"
# 中文1个字符大于1个token,不可设置为模型最大上下文
CHUNK_SIZE = 15000 
OVERLAP = 500      

def split_text(text):
    """ 滑动窗口切片逻辑 """
    chunks, start = [], 0
    if not text: return [""]
    while start < len(text):
        chunks.append(text[start : start + CHUNK_SIZE])
        start += (CHUNK_SIZE - OVERLAP)
        if start >= len(text): break
    return chunks

async def analyze_text_task_stream(content, target_output, task_type="ANALYSIS"):
    """
    核心引擎:支持分析(ANALYSIS)、汇总(SUMMARY)、对比(COMPARISON)
    引擎中有提示词,参数设置,和异步httpx调用
    """
    prompts = {
        "ANALYSIS": "你是一个严格受控的深度分析引擎。思考步骤：1.识别核心事件 2.分析语气情绪 3.提炼关键词。",
        "SUMMARY": "你是一个报告汇总专家。思考步骤：1.对比各段异同 2.去重并保留关键细节 3.合成连贯摘要。",
        "COMPARISON": "你是一个首席审计分析官。思考步骤：1.识别各文件间的立场矛盾 2.归纳共有信息 3.评估各文件的独特价值。"
    }
    
    few_shot = (
        "\n--- 示例 ---\n"
        "待处理：最近体验了 ROCm 方案，安装有坑但速度提升 40%。\n"
        "输出：\n<Thought>提及技术方案与性能，有负面细节但总体评价积极。</Thought>\n"
        "## 📝 核心摘要\nROCm 方案虽安装波折但性能提升显著，评价积极。\n"
        "## 🎭 情感倾向\n积极\n"
        "## 🔑 核心关键词\nROCm, 加速, 性能\n"
        "--- 示例结束 ---\n\n"
    )

    instruction = (
        f"{prompts.get(task_type, prompts['ANALYSIS'])}\n"
        # 仅在基础分析时注入示例
        f"{few_shot if task_type == 'ANALYSIS' else ''}"
        "**必须遵循的输出格式：**\n"
        "<Thought>\n[在此记录你的对比/推理逻辑]\n</Thought>\n\n"
    )
    
    if task_type == "COMPARISON":
        instruction += "## 📊 全局对比综述\n[总结整体情况]\n## 🤝 共同核心点\n[列出一致的信息]\n## 🔄 主要差异分析\n[分析立场或数据的冲突]"
    else:
        instruction += "## 📝 核心摘要\n[100字以内]\n## 🎭 情感倾向\n[积极/中性/消极]\n## 🔑 核心关键词\n[3个词,逗号分隔]"

    payload = {
        "model": MODEL_NAME,
        "prompt": f"{instruction}\n\n待处理内容：\n{content}",
        "stream": True,
        "options": {"num_ctx": 32000, "temperature": 0.0}
    }

    full_response, buffer = "", bytearray()

    # 增加超时时间以应对复杂的对比任务
    async with httpx.AsyncClient(timeout=300) as client:
        try:
            async with client.stream("POST", OLLAMA_URL, json=payload) as response:
                if response.status_code != 200:
                    target_output.append_stdout(f"\n❌ API 错误: {response.status_code}")
                    return False, f"Error {response.status_code}"
                # 异步方式逐个字节响应,每个 chunk 大小不固定
                async for chunk in response.aiter_bytes():
                    buffer.extend(chunk)
                    # 单个 chunk 可能包含多个换行符分割的行
                    while b'\n' in buffer:
                        line, rest = buffer.split(b'\n', 1)
                        buffer = bytearray(rest)
                        try:
                            line_json = json.loads(line.decode('utf-8'))
                            part = line_json.get("response", "")
                            target_output.append_stdout(part)
                            full_response += part
                        except: continue
            return (True, full_response) if full_response else (False, "内容为空")
        except Exception as e:
            target_output.append_stdout(f"\n❌ 通信异常: {str(e)}")
            return False, str(e)

async def process_single_file(filename, content, target_output):
    """ 单文件级联处理流水线 """
    target_output.append_stdout(f"\n\n📄 正在处理: {filename} ({len(content)} 字)...\n")
    chunks = split_text(content)
    chunk_reports = []
    
    for i, chunk in enumerate(chunks):
        if len(chunks) > 1: 
            target_output.append_stdout(f"\n[分段 {i+1}/{len(chunks)}] ")
        success, res = await analyze_text_task_stream(chunk, target_output, task_type="ANALYSIS")
        if not success: return None
        chunk_reports.append(res)
    
    # 级联汇总:如果文件内有多个切片,汇总总结
    if len(chunk_reports) > 1:
        target_output.append_stdout(f"\n\n🔄 正在汇总文件【{filename}】的多个分段...\n")
        success, final_sum = await analyze_text_task_stream("\n".join(chunk_reports), target_output, task_type="SUMMARY")
        return final_sum if success else None
    else:
        return chunk_reports[0]

async def run_global_comparison(b):
    """ 按钮逻辑,包含单文件处理和多文件处理 """
    output.clear_output()
    files_count = len(uploader.value)

    if files_count < 1:
        output.append_stdout("⚠️ 请先上传文件。")
        return
    
    output.append_stdout(f"🚀 启动流水线：共 {files_count} 个文件\n")
    all_file_reports = []
    
    # 单文件内部切片与汇总
    for file_info in uploader.value:
        fname = file_info['name']
        raw_data = file_info['content']

        # 使用 tobytes 方法解码
        try:
            text = raw_data.tobytes().decode('utf-8')
        except:
            output.append_stdout(f"❌ 文件 {fname} 解码失败\n")
            continue
            
        report = await process_single_file(fname, text, output)
        if report:
            all_file_reports.append(f"### 文件源：{fname}\n{report}")
            output.append_stdout(f"\n\n✅ {fname} 局部处理完成。\n" + "-"*30)

    # 如果多文件,执行全局对比分析
    if len(all_file_reports) >= 2:
        output.append_stdout("\n" + "█"*15 + " 正在生成全局对比报告 " + "█"*15 + "\n\n")
        comparison_input = "\n\n".join(all_file_reports)
        success, detail = await analyze_text_task_stream(comparison_input, output, task_type="COMPARISON")
        
        if success:
            output.append_stdout(f"\n\n{'='*40}\n✨ 所有文件横向对比分析已完成！")
        else:
            output.append_stdout(f"\n\n❌ 对比失败：{detail}")
    else:
        output.append_stdout("\n💡 文件数量较少，已跳过横向对比分析。")

# 用户界面 UI 逻辑
uploader = widgets.FileUpload(accept='.txt,.md', multiple=True, description="选择多文件")
start_btn = widgets.Button(description="启动对比分析流水线", button_style='primary', layout=widgets.Layout(width='200px'))
# 使用 asyncio 确保不会阻塞主线程 UI
start_btn.on_click(lambda b: asyncio.create_task(run_global_comparison(b)))
output = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<h3>跨文档智能审计与对比系统 (Global Comparison)</h3>"),
    uploader, 
    start_btn, 
    output
]))